# 03 — Serve Qwen3.5-4B (bitsandbytes 4-bit) + the QLoRA adapter with vLLM on a T4

vLLM's speed comes from **PagedAttention + continuous batching on a GPU**: it shines when many
requests run together. So this notebook measures **throughput at 1, 8 and 32 concurrent prompts**,
not just one request.

Run in a **fresh** T4 runtime (vLLM pins its own torch). The adapter comes from notebook 02 (Drive).

> **Check first:** Qwen3.5 is a hybrid model (linear attention + full attention). vLLM support for
> this architecture on a T4 (compute capability 7.5, no FlashAttention-2), and for bitsandbytes with it,
> depends on the vLLM version. If loading fails, the cell prints the error: set
> `MODEL_ID = "Qwen/Qwen3-4B"` (a standard transformer) to still see the vLLM batching effect.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U vllm bitsandbytes

In [ ]:
# Save everything to Google Drive: free Colab sessions disconnect, and notebook 03 needs the adapter.
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/model_quantization"
import os; os.makedirs(WORK, exist_ok=True)
print("Saving to", WORK)

In [ ]:
import torch, vllm
print("vllm", vllm.__version__, "| torch", torch.__version__, "|", torch.cuda.get_device_name(0))

MODEL_ID = "Qwen/Qwen3.5-4B"
ADAPTER_DIR = f"{WORK}/qwen3.5-4b-nf4-dq-qlora-adapter"   # from notebook 02 (set None to skip LoRA)
import os
if ADAPTER_DIR and not os.path.isdir(ADAPTER_DIR):
    print("No adapter found - running the base model only"); ADAPTER_DIR = None

In [ ]:
# Same CSV columns as the laptop API's logs/requests.csv, so the results can be compared directly.
import csv, time, uuid
from datetime import datetime
from pathlib import Path

LOG_CSV = Path(WORK) / "colab_requests.csv"
COLUMNS = ["request_id", "model", "model_name", "start_time", "end_time", "duration_sec", "duration_min",
           "prompt", "response", "prompt_tokens", "new_tokens", "tokens_per_second", "status", "error"]

def now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]

def log_row(row):
    new = not LOG_CSV.exists()
    with LOG_CSV.open("a", newline="", encoding="utf-8-sig" if new else "utf-8") as f:
        w = csv.DictWriter(f, fieldnames=COLUMNS, quoting=csv.QUOTE_ALL, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow({c: row.get(c, "") for c in COLUMNS})

In [ ]:
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

t0 = time.perf_counter()
llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",   # 4-bit weights on load (bitsandbytes NF4)
    dtype="half",                  # T4: no bfloat16
    max_model_len=2048,
    gpu_memory_utilization=0.85,   # the rest of the 15 GB becomes PagedAttention KV blocks
    enable_lora=ADAPTER_DIR is not None,
    max_lora_rank=16,
    enforce_eager=True,            # skip CUDA-graph capture: faster start-up, more robust on T4
)
print(f"vLLM ready in {time.perf_counter() - t0:.0f} s")
lora = LoRARequest("qlora", 1, ADAPTER_DIR) if ADAPTER_DIR else None
tokenizer = llm.get_tokenizer()

In [ ]:
def chat_prompt(p):
    return tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                         add_generation_prompt=True, enable_thinking=False)

def run_batch(prompts, max_tokens=128, use_lora=False, label="vllm-bnb-t4"):
    """Send all prompts at once (vLLM batches them), log each one, return totals."""
    params = SamplingParams(temperature=0.0, max_tokens=max_tokens)
    start, t0 = now(), time.perf_counter()
    outs = llm.generate([chat_prompt(p) for p in prompts], params, lora_request=lora if use_lora else None)
    secs, end = time.perf_counter() - t0, now()
    rid = uuid.uuid4().hex[:12]
    total = 0
    for p, o in zip(prompts, outs):
        n = len(o.outputs[0].token_ids); total += n
        log_row(dict(request_id=rid, model=label + ("-qlora" if use_lora else ""), model_name=MODEL_ID,
                     start_time=start, end_time=end, duration_sec=round(secs, 2), duration_min=round(secs / 60, 3),
                     prompt=p, response=o.outputs[0].text.strip(), prompt_tokens=len(o.prompt_token_ids),
                     new_tokens=n, tokens_per_second=round(n / secs, 2), status="ok"))
    return {"concurrent": len(prompts), "seconds": round(secs, 2), "new_tokens": total,
            "total_tok_per_s": round(total / secs, 1), "per_request_tok_per_s": round(total / secs / len(prompts), 1)}

out = llm.generate([chat_prompt("What is NF4 quantisation? One sentence.")], SamplingParams(temperature=0, max_tokens=40))
print(out[0].outputs[0].text)

## Throughput: 1 vs 8 vs 32 requests at once
One request at a time is limited by reading all weights for every token. With batching, every weight
read serves many requests, so **total tokens/s grows almost linearly** until the GPU is busy.

In [ ]:
import pandas as pd
QUESTIONS = ["What is NF4 quantisation?", "Explain PagedAttention.", "Give three tips for staying healthy.",
             "What is a KV cache?", "Explain LoRA in two sentences.", "Why is the sky blue?",
             "Write a haiku about GPUs.", "What is double quantisation?"]
run_batch(QUESTIONS[:1], max_tokens=16)  # warm-up
results = [run_batch((QUESTIONS * 4)[:n]) for n in (1, 8, 32)]
pd.DataFrame(results)

In [ ]:
if lora:
    print(pd.DataFrame([run_batch(QUESTIONS, use_lora=False), run_batch(QUESTIONS, use_lora=True)],
                       index=["base", "QLoRA adapter"]))
print("CSV log:", LOG_CSV)

## Optional: OpenAI-compatible server
Instead of the Python API above you can run a server (in a new runtime, or after `del llm`):

```
!nohup vllm serve Qwen/Qwen3.5-4B --quantization bitsandbytes --dtype half --max-model-len 2048 \
    --enable-lora --lora-modules qlora=/content/drive/MyDrive/model_quantization/qwen3.5-4b-nf4-dq-qlora-adapter \
    --port 8000 > vllm.log 2>&1 &
```
Then `POST http://localhost:8000/v1/chat/completions` with `"model": "qlora"`. To call it from the laptop's
Streamlit app you would need a tunnel (e.g. cloudflared), since Colab has no public port.